# main.ipynb


Welcome! This notebook is a step-by-step walkthrough of the data analysis in the paper. The raw data is provided seperately from the respository with its ow DOI. Download the data and adjust the path2data variable below to point to the correct directory.

The code first starts by initializing a dictionary that stores results analysed results per KID.
The raw data provided is only of KID26, if other data is desired, please contact the author.


### Install dependencies

Activate a virtual environment and open a terminal
run 'pip install -r requirements.txt' to install all the required dependencies

### Load modules

In [1]:
from scripts.initialize_kids import initialize_kid26
from scripts.fit_quality_factors import *
from scripts.stack_filters import *
from scripts.plot_pulse import photon_timestreams
from scripts.plot_noise import noise_spectra
from scripts.analyse_resolving_powers import pulse_height_distribution, plot_distributions, plot_resolving_powers
from scripts.analyse_efficiency import counts_vs_temps, fit_efficiency
from scripts.analyse_dark_countrates import load_dark_counts, analyse_coincidences, analyse_dark_countrates

from scripts.utils import matplotlibcolors                  # defines custom colors for plotting
plt.style.use('scripts/utils/matplotlibrc')                 # custom styleguide for plots

# magic command to plot figures as widgets with zoom options and such
%matplotlib widget                                          

### Set data directory

In [2]:
path2data = 'D:/Data/'              # adjust this to the directory where the raw data is stored

### Initialize kid_dict

In [ ]:
name = 'KID26'                      # the general KID identifier, in between different measurement the KID number might change
from_scratch = False                               # if True, the kid_dict will be initialized empty and all results contained in kid_dict are lost
initialize_kid26(path2data, name, from_scratch)    # this functions loads the general settings of KID26

In [ ]:
initialize_quality_factors(path2data, name)         # this function fits the quality factors of all the measurements

In [ ]:
initialize_filterstacks(path2data, name)            # this functions generates the filterstacks used for the experiments

### Plot noise and pulses

In [ ]:
noise_spectra(path2data, name)   # calculate and plot the noise spectra of all measurements

In [ ]:
photon_timestreams(path2data, name)    # plot the photon timestreams together from all measurements

### Analyse energy resolving power (skip if already analysed)

We analyse the energy resolving power at 4 different wavelengths: 3.8, 8.5, 18.5 and 25 um. Below these are analysed per wavelength. Some general settings can be specified.

In [ ]:
wl = '3.8um'                        # measurement identifier, this measurement contains 300s of data
thresholds = [3, 22]                # list of pulse detection thresholds, the data is analysed for each of these thresholds. The first value is the considered as the noise threshold, the last value as the direct pulse detection threshold
chuncks = [None, None]              # list of number of chuncks (chuncksize is 50s) to be analysed per detection threshold, if None, all chuncks are analysed
lifetime = 55                       # lifetime for the exponential filter, this slightly differs per setup and is iteretively determined
fit_tqp = [100, 250]                # range for fitting the exponetial decay in microseconds 
secondary_filter = .25              # pulse height fraction compared to the average pulse height for secondary pulse rejection, input value between 0 and 1
binsize = 0.025                     # binsize of the pulseheight distributions

pulse_height_distribution(path2data, name, wl, thresholds, chuncks, lifetime, fit_tqp, secondary_filter, binsize)

In [ ]:
wl = '8.5um'                    # this measurement contains 200s of data
thresholds = [3, 20]              
chuncks = [None, None]            
lifetime = 55                 
fit_tqp = [100, 250]          
secondary_filter = .25      
binsize = 0.025

pulse_height_distribution(path2data, name, wl, thresholds, chuncks, lifetime, fit_tqp, secondary_filter, binsize)

In [ ]:
wl = '18.5um'                    # this measurement contains 40s of data
thresholds = [3, 9]               
chuncks = [None, None]            
lifetime = 70                 
fit_tqp = [50, 250]   
secondary_filter = .5       
binsize = 0.07

pulse_height_distribution(path2data, name, wl, thresholds, chuncks, lifetime, fit_tqp, secondary_filter, binsize)

In [ ]:
wl = '25um'                         # this measurement contains 40s of data
thresholds = [3, 8]               
chuncks = [None, None]           
lifetime = 75                 
fit_tqp = [100, 250]           
secondary_filter = .5
binsize = 0.07

pulse_height_distribution(path2data, name, wl, thresholds, chuncks, lifetime, fit_tqp, secondary_filter, binsize)

### Plot energy resolving powers

In [ ]:
plot_distributions(path2data, name)                     # this functions plots the pulsle height distributions at all wavelengths
plot_resolving_powers(path2data, name)                  # this function plots the energy resolving powers at all wavelengths

### Analyse dark count rates

Here we analyse the 10000s multichannel dark measurement to determine the dark count rates per wavelength. The raw data is over 300Gb and thus not included in the reproduction package. The data has been preprocessed in segments of 10s. This preprocessed data is made avaiable and is compiled and further analysed in the following. 

In [ ]:
wl = 'mux'                                                              # measurement identifier                                  
load_dark_counts(path2data, name, wl, from_scratch=False)               # compile all the preprocessed data

coincidence_dt = 40e-6                                                  # sets the time window for coincidence detection in seconds
analyse_coincidences(path2data, name, coincidence_dt)                   # this function analyses the coincidence events in the dark count data and outputs two figures

confidence_intervals = [3, 4, 5]                                        # confidence intervals used to compute a dark count rate using the energy resolving power 
analyse_dark_countrates(path2data, name, wl, confidence_intervals)      # function to compute the dark count rates per wavelength and plots the results

### Efficiency

An efficiency measurement is done at 18.5um as explained in the paper. The photonrates are measured as a function of radiator temperature and a model is fitted to extract the efficiency.

In [ ]:
wl = '18.5um'                                                       # measurement identifier
temps = [3, 30, 50, 85, 100, 135, 142, 150, 160, 180]               # measured radiator temperatures in K
lifetime = 70                                                       # lifetime for the exponential filter, this set equal to the one used for to determine the energy resolving power 
from_scratch = False                                                # set to True to redo the pulse detection as a function of temperature, if False the existing results are used
counts_vs_temps(path2data, name, wl, temps, lifetime, from_scratch) # this function determines the photon countrates as a function of the radiator temperature and stores these in the kid_dict
fit_efficiency(path2data, name, wl, [3, None])                      # this functions determines the efficiency by fitting a model to the measured photonrates as a function of temperature